# Conquer Mobile — Asset Preparation Pipeline (One Cell)

خلية كود واحدة تنزّل الأصول والتقارير من إصدار GitHub، تتحقق من ZIP، تستخرج بأمان، تصنّف النماذج والصور، تقترح روابط الخامات، تنشئ Manifest وحزمة منظمة، ثم تنشر المخرجات وتتحقق منها.

لا Drive ولا GPU ولا اختيار ملفات أو إدخال مسارات. لا تعدّل ملفات مشروع اللعبة. روابط الخامات استدلالية وتحتاج مراجعة بصرية، ولا يوجد اختبار استيراد فعلي في Godot.

In [ ]:
%pip -q install requests pillow
import os,re,csv,json,shutil,hashlib,zipfile,stat,requests
from pathlib import Path,PurePosixPath
from collections import Counter
from google.colab import userdata

OWNER="ZzwahwahzZ"; REPO="Conquer-Mobile-Assets."; TAG="mobile-extracted-assets"
BUNDLE="extracted-assets-001.zip"; EXPECTED_SHA256="0d21a110c09b762ed5f1a5f4da022857b0534e489eb85052d15da460cb229d2a"; API=f"https://api.github.com/repos/{OWNER}/{REPO}"
W=Path("/content/conquer_mobile_asset_prep"); SRC=W/"source.zip"; RAW=W/"raw"; OUT=W/"prepared"
ME={".glb",".gltf",".obj",".fbx",".dae",".blend",".3ds",".stl",".ply"}
IE={".png",".jpg",".jpeg",".webp",".tga",".bmp",".dds",".tif",".tiff"}
CHUNK=8*1024*1024

def sha(p):
 h=hashlib.sha256()
 with open(p,"rb") as f:
  for b in iter(lambda:f.read(CHUNK),b""): h.update(b)
 return h.hexdigest()

def extract(zp,dst):
 dst=dst.resolve(); n=total=0
 with zipfile.ZipFile(zp) as z:
  bad=z.testzip()
  if bad: raise RuntimeError("ZIP CRC error: "+bad)
  for x in z.infolist():
   q=PurePosixPath(x.filename.replace("\\","/"))
   if q.is_absolute() or ".." in q.parts or not q.parts: raise RuntimeError("Unsafe ZIP path "+x.filename)
   if stat.S_ISLNK((x.external_attr>>16)&0xffff): raise RuntimeError("ZIP symlink rejected")
   p=(dst/Path(*q.parts)).resolve()
   if dst not in p.parents and p!=dst: raise RuntimeError("ZIP path traversal")
   if x.is_dir(): p.mkdir(parents=True,exist_ok=True); continue
   p.parent.mkdir(parents=True,exist_ok=True)
   with z.open(x) as a,open(p,"wb") as b: shutil.copyfileobj(a,b,CHUNK)
   n+=1; total+=x.file_size
 return n,total

def cat(p):
 s=(p.parent.name+" "+p.stem).lower()
 rules={"characters":["character","player","role","body","hair","head","face"],"monsters":["monster","mob","boss","beast","dragon","demon"],"mounts":["mount","horse","ride","steed"],"maps":["map","terrain","scene","world","city","building","tree","rock"],"effects":["effect","skill","magic","particle","fire","lightning","aura"],"ui":["icon","ui","button","interface"],"items":["item","armor","armour","sword","blade","bow","shield","helmet","ring"]}
 for k,ws in rules.items():
  if any(w in s for w in ws): return k
 return "unclassified"

def toks(p): return set(x for x in re.split(r"[^a-z0-9]+",p.stem.lower()) if len(x)>2)
def clean(s): return re.sub(r"[^A-Za-z0-9._-]+","_",s).strip("._")[:130] or "asset"

def main():
 if W.exists(): shutil.rmtree(W)
 RAW.mkdir(parents=True); OUT.mkdir(parents=True)
 token=userdata.get("GITHUB_TOKEN")
 if not token: raise RuntimeError("Missing Colab Secret GITHUB_TOKEN")
 s=requests.Session(); s.headers.update({"Authorization":"Bearer "+token,"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"})
 print("Resolving fixed GitHub release...")
 r=s.get(f"{API}/releases/tags/{TAG}",timeout=60); r.raise_for_status(); rel=r.json()
 au=f'{API}/releases/{rel["id"]}/assets'; r=s.get(au,timeout=60); r.raise_for_status(); assets=r.json()
 names={a["name"]:a for a in assets}
 required=[BUNDLE,"deep-3d-asset-audit.json","deep-3d-asset-audit.csv"]
 if any(x not in names for x in required): raise RuntimeError("Missing source bundle or deep audit reports")
 src=names[BUNDLE]
 print("Downloading",BUNDLE,src["size"],"bytes")
 with s.get(src["browser_download_url"],stream=True,timeout=(60,1200)) as r:
  r.raise_for_status()
  with open(SRC,"wb") as f:
   for b in r.iter_content(CHUNK):
    if b:f.write(b)
 if SRC.stat().st_size!=src["size"]: raise RuntimeError("Source archive size mismatch")
 digest=sha(SRC)
 if digest!=EXPECTED_SHA256: raise RuntimeError("Source SHA-256 mismatch: "+digest)
 print("DOWNLOAD VERIFIED",SRC.stat().st_size,"SHA256",digest)
 n,total=extract(SRC,RAW); print("EXTRACTED",n,"files",total,"bytes")
 for nm in required[1:]:
  r=s.get(names[nm]["browser_download_url"],timeout=180); r.raise_for_status()
  p=W/nm;p.write_bytes(r.content)
  if p.stat().st_size!=names[nm]["size"]: raise RuntimeError(nm+" size mismatch")
 audit=json.loads((W/required[1]).read_text(encoding="utf-8"))
 rows=[]
 with open(W/required[2],newline="",encoding="utf-8-sig") as f: rows=list(csv.DictReader(f))
 auditmap={(x.get("path") or x.get("file") or x.get("relative_path") or "").replace("\\","/"):x for x in rows}
 files=[p for p in RAW.rglob("*") if p.is_file()]
 models=sorted(p for p in files if p.suffix.lower() in ME); imgs=sorted(p for p in files if p.suffix.lower() in IE)
 print("Targets:",len(models),"models |",len(imgs),"images/textures")
 for d in ["models","textures","reports"]: (OUT/d).mkdir(parents=True,exist_ok=True)
 for c in ["characters","monsters","mounts","maps","effects","items","ui","unclassified"]: (OUT/"models"/c).mkdir(parents=True,exist_ok=True)
 image_info=[{"source_path":p.relative_to(RAW).as_posix(),"name":p.name,"tokens":sorted(toks(p))} for p in imgs]
 manifest=[]; linked=set()
 for i,m in enumerate(models,1):
  relp=m.relative_to(RAW).as_posix(); category=cat(m); dest=OUT/"models"/category/(clean(m.stem)+m.suffix.lower())
  if dest.exists(): dest=dest.with_name(clean(m.stem)+"_"+hashlib.sha1(relp.encode()).hexdigest()[:8]+m.suffix.lower())
  shutil.copy2(m,dest); mt=toks(m); candidates=[]
  for im in imgs:
   it=toks(im); common=mt&it; score=1.0 if m.stem.lower()==im.stem.lower() else (len(common)/max(1,len(mt|it)) if common else 0)
   if common and m.parent==im.parent: score=min(.99,score+.18)
   if score>0: candidates.append((score,im))
  candidates.sort(key=lambda x:(-x[0],x[1].as_posix()))
  matches=[]
  for score,im in candidates[:5]:
   confidence="high" if score>=.75 else "medium" if score>=.4 else "low"
   matches.append({"texture":im.relative_to(RAW).as_posix(),"score":round(score,3),"confidence":confidence})
   if score>=.4: linked.add(im)
  manifest.append({"id":f"asset_{i:03d}","source_path":relp,"prepared_path":dest.relative_to(OUT).as_posix(),"filename":m.name,"format":m.suffix.lower(),"category":category,"bytes":m.stat().st_size,"sha256":sha(m),"audit_record":auditmap.get(relp,{}),"texture_candidates":matches,"texture_match_status":"candidate_found" if any(x["confidence"] in ("high","medium") for x in matches) else "visual_review_required","godot_hint":"GLB/GLTF or OBJ import candidate; validate all formats in target Godot editor"})
  if i%20==0 or i==len(models): print("Prepared models",i,"/",len(models))
 for i,im in enumerate(imgs,1):
  relp=im.relative_to(RAW).as_posix(); d=OUT/"textures"/cat(im);d.mkdir(parents=True,exist_ok=True)
  dest=d/(clean(im.stem)+im.suffix.lower())
  if dest.exists():dest=dest.with_name(clean(im.stem)+"_"+hashlib.sha1(relp.encode()).hexdigest()[:8]+im.suffix.lower())
  shutil.copy2(im,dest); image_info[i-1].update({"prepared_path":dest.relative_to(OUT).as_posix(),"bytes":im.stat().st_size,"sha256":sha(im),"candidate_model_ids":[x["id"] for x in manifest if any(y["texture"]==relp for y in x["texture_candidates"])]})
 for nm in ["model-contact-sheet.jpg","deep-3d-asset-audit.html","deep-3d-asset-audit.json","deep-3d-asset-audit.csv"]:
  if nm in names:
   r=s.get(names[nm]["browser_download_url"],timeout=180);r.raise_for_status();(OUT/"reports"/nm).write_bytes(r.content)
 report={"source_release":rel["html_url"],"source_archive":BUNDLE,"source_sha256":sha(SRC),"extracted_files":n,"extracted_bytes":total,"model_count":len(models),"image_texture_count":len(imgs),"category_counts":dict(Counter(x["category"] for x in manifest)),"format_counts":dict(Counter(x["format"] for x in manifest)),"texture_matching":{"models_with_medium_high_candidates":sum(x["texture_match_status"]=="candidate_found" for x in manifest),"unique_images_with_medium_high_candidates":len(linked),"method":"filename/folder heuristic only; not confirmed material binding"},"rigging_animation_note":"Prior deep audit found zero rigged GLBs and zero animated models; animation authoring/rigging is still required.","godot_validation":"No Godot editor import or runtime test is performed in this pipeline.","models":manifest,"images":image_info}
 mp=OUT/"reports"/"asset-preparation-manifest.json";mp.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
 cp=OUT/"reports"/"asset-preparation-manifest.csv"
 with open(cp,"w",newline="",encoding="utf-8-sig") as f:
  w=csv.DictWriter(f,fieldnames=["id","source_path","prepared_path","filename","format","category","bytes","sha256","texture_match_status","texture_candidates","godot_hint"]);w.writeheader()
  for x in manifest:
   w.writerow({"id":x["id"],"source_path":x["source_path"],"prepared_path":x["prepared_path"],"filename":x["filename"],"format":x["format"],"category":x["category"],"bytes":x["bytes"],"sha256":x["sha256"],"texture_match_status":x["texture_match_status"],"texture_candidates":json.dumps(x["texture_candidates"],ensure_ascii=False),"godot_hint":x["godot_hint"]})
 (OUT/"README-ASSET-PACK.md").write_text("# Prepared asset pack\n\nNon-destructive organized copy of extracted source assets.\n\n- models/: categorized model files\n- textures/: all extracted image/texture files\n- reports/: model manifest, texture candidates and previous audit reports\n\nTexture links are heuristic candidates, not confirmed material assignments. Rigging/animation and Godot import/runtime validation remain separate tasks. Original game project files are not modified.\n",encoding="utf-8")
 summary={k:v for k,v in report.items() if k not in ("models","images")}
 summary["prepared_model_count"]=len(manifest);summary["prepared_image_count"]=len(image_info)
 sp=W/"asset-preparation-summary.json";sp.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
 package=W/"conquer-mobile-prepared-assets.zip"
 with zipfile.ZipFile(package,"w",zipfile.ZIP_DEFLATED,compresslevel=3,allowZip64=True) as z:
  for p in OUT.rglob("*"):
   if p.is_file():z.write(p,p.relative_to(OUT).as_posix())
 print("PACKAGE CREATED",package.stat().st_size,"bytes")
 upload=[package,sp,mp,cp,OUT/"README-ASSET-PACK.md"]; up=rel["upload_url"].split("{",1)[0]
 for p in upload:
  r=s.get(au,timeout=60);r.raise_for_status();old=next((x for x in r.json() if x["name"]==p.name),None)
  if old:
   d=s.delete(old["url"],timeout=60);d.raise_for_status()
  with open(p,"rb") as f:r=s.post(up,params={"name":p.name},headers={**s.headers,"Content-Type":"application/octet-stream"},data=f,timeout=(60,1800))
  r.raise_for_status();a=r.json()
  if a.get("name")!=p.name or a.get("size")!=p.stat().st_size:raise RuntimeError("Upload verification failed "+p.name)
  print("UPLOADED + VERIFIED",p.name,a["size"])
 r=s.get(au,timeout=60);r.raise_for_status(); final={x["name"]:x for x in r.json()}
 for p in upload:
  if p.name not in final or final[p.name]["size"]!=p.stat().st_size:raise RuntimeError("Release verification failed "+p.name)
 print("SUCCESS: prepared assets package and four reports uploaded and verified.")
 print("Release:",rel["html_url"])
 print("Models:",len(models),"Images:",len(imgs),"Texture candidates:",len(linked))
 print("No game project files modified; heuristic texture links require visual confirmation.")

main()
